## Loading Dataset

In [ ]:
import qnt.data as qndata
import pandas as pd
import numpy as np
import xarray as xr

data = qndata.cryptodaily_load_data(
    min_date="2015-01-01"
)

<xarray.DataArray 'cryptodaily' (field: 6, time: 4177, asset: 76)> Size: 15MB
array([[[           nan,            nan,            nan, ...,
                    nan,            nan,            nan],
        [           nan,            nan,            nan, ...,
                    nan,            nan,            nan],
        [           nan,            nan,            nan, ...,
                    nan,            nan,            nan],
        ...,
        [6.22800000e+01, 1.56700000e-01, 9.34000000e-02, ...,
         8.94000000e-03,            nan,            nan],
        [6.09700000e+01, 1.57300000e-01, 9.22000000e-02, ...,
         8.59000000e-03,            nan,            nan],
        [6.37200000e+01, 1.65400000e-01, 9.45000000e-02, ...,
                    nan,            nan,            nan]],

       [[           nan,            nan,            nan, ...,
                    nan,            nan,            nan],
        [           nan,            nan,            nan, ...,
                    nan,            nan,            nan],
        [           nan,            nan,            nan, ...,
                    nan,            nan,            nan],
...
         3.40454220e+07,            nan,            nan],
        [2.06462414e+08, 5.45023352e+08, 3.66455531e+07, ...,
         6.19503357e+06,            nan,            nan],
        [1.45548920e+08, 4.86559298e+08, 3.37437097e+07, ...,
                    nan,            nan,            nan]],

       [[           nan,            nan,            nan, ...,
                    nan,            nan,            nan],
        [           nan,            nan,            nan, ...,
                    nan,            nan,            nan],
        [           nan,            nan,            nan, ...,
                    nan,            nan,            nan],
        ...,
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00,            nan,            nan],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00,            nan,            nan],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
                    nan,            nan,            nan]]],
      shape=(6, 4177, 76))
Coordinates:
  * field    (field) object 48B 'open' 'low' 'high' 'close' 'vol' 'is_liquid'
  * time     (time) datetime64[ns] 33kB 2015-01-01 2015-01-02 ... 2026-06-08
  * asset    (asset) object 608B 'AAVE' 'ADA' 'ALGO' ... 'STRAX' 'WAVES' 'XEM'

## Dataset Metadata

In [ ]:
print(data.dims)

print("\nAssets")
print(list(data.asset.values))

print("\nFields")
print(list(data.field.values))

Dimensions
('field', 'time', 'asset')

Assets
['AAVE', 'ADA', 'ALGO', 'APT', 'AR', 'ARB', 'ATOM', 'AVAX', 'BCH', 'BGB', 'BNB', 'BONK', 'BSV', 'BTC', 'CC', 'CRO', 'DASH', 'DOGE', 'DOT', 'ENA', 'ETC', 'ETH', 'FARTCOIN', 'FET', 'FIL', 'FLOKI', 'FTT', 'GRT', 'HBAR', 'HYPE', 'ICP', 'IMX', 'INJ', 'JUP', 'KAS', 'LEO', 'LINK', 'LTC', 'LUNA', 'MNT', 'NEAR', 'NEO', 'OKB', 'ONDO', 'OP', 'PEPE', 'PI', 'POL', 'RENDER', 'RUNE', 'SHIB', 'SOL', 'STX', 'SUI', 'TAO', 'THETA', 'TON', 'TRUMP', 'TRX', 'UNI', 'VET', 'WIF', 'XLM', 'XMR', 'XRP', 'XTZ', 'ZEC', 'BTS', 'EOS', 'GNT', 'IOTA', 'LSK', 'STEEM', 'STRAX', 'WAVES', 'XEM']

Fields
['open', 'low', 'high', 'close', 'vol', 'is_liquid']

## Asset History

In [ ]:
close = data.sel(field="close")

summary = []

for asset in close.asset.values:

    series = close.sel(asset=asset)

    valid = series.dropna("time")

    if len(valid.time) > 0:

        summary.append({
            "Asset": asset,
            "Start": str(valid.time.min().values)[:10],
            "End": str(valid.time.max().values)[:10],
            "Days": len(valid.time)
        })

pd.DataFrame(summary)

	Asset	Start	End	Days
0	AAVE	2020-12-15	2026-06-08	2001
1	ADA	    2017-10-02	2026-06-08	3172
2	ALGO	2019-06-19	2026-06-08	2547
3	APT	    2022-11-02	2026-06-08	1315
4	AR	    2020-09-26	2026-06-08	2082
...	...	     ...	...	...
71	LSK	    2016-06-06	2026-06-07	3411
72	STEEM	2016-04-19	2026-06-07	3577
73	STRAX	2016-08-12	2026-06-07	3526
74	WAVES	2016-06-03	2024-06-17	2802
75	XEM	    2015-04-01	2024-06-17	3233
76 rows × 4 columns

## Universe Construction

In [ ]:
is_liquid = data.sel(field="is_liquid")

liquid_count = is_liquid.sum("asset")

print("Average liquid assets")
print(float(liquid_count.mean()))

print()

print("Minimum")
print(float(liquid_count.min()))

print()

print("Maximum")
print(float(liquid_count.max()))

Average liquid assets
9.796025855877424

Minimum
7.0

Maximum
10.0

## Universe Turnover

In [ ]:
turnover = abs(
    is_liquid.diff("time")
).sum("asset")

print(
    "Average Daily Turnover:",
    float(turnover.mean())
)

Average Daily Turnover: 0.04525862068965517

## Return Destribution

In [ ]:
ret = close / close.shift(time=1) - 1

is_liquid = data.sel(field="is_liquid")

ret = ret.where(is_liquid == 1)

r = ret.values.flatten()

r = r[np.isfinite(r)]

print("Mean Daily Return")
print(np.mean(r))

print()

print("Std Daily Return")
print(np.std(r))

print()

import scipy.stats as stats

print("Skewness")
print(stats.skew(r))

print()

print("Kurtosis")
print(stats.kurtosis(r))

print()

print("Largest Gain")
print(np.max(r))

print()

print("Largest Loss")
print(np.min(r))

Mean Daily Return
0.002270062453427016

Std Daily Return
0.06035949803107136

Skewness
3.067921682431499

Kurtosis
67.78788940850548

Largest Gain
1.835286795975521

Largest Loss
-1.0

## Volatility Characteristics

In [ ]:
ret = close / close.shift(time=1) - 1

ret = ret.where(
    data.sel(field="is_liquid") == 1
)

ann_vol = ret.std("time") * np.sqrt(365)

vol_df = ann_vol.to_pandas().sort_values(
    ascending=False
)

print(vol_df.head(15))

print("\nAverage Annualized Volatility:")
print(vol_df.mean())

asset
STEEM    3.063000
ZEC      2.748353
ICP      2.362985
IOTA     1.898537
XEM      1.799704
NEO      1.720280
PI       1.524055
UNI      1.347208
XMR      1.334845
THETA    1.312995
XTZ      1.299835
BCH      1.286524
WAVES    1.280943
XRP      1.248086
DASH     1.208533
Name: cryptodaily, dtype: float64

Average Annualized Volatility:
1.2562258712013987

## Correlation Structure

In [ ]:
ret = close / close.shift(time=1) - 1

ret = ret.where(
    data.sel(field="is_liquid") == 1
)

corr_df = ret.to_pandas()

corr_matrix = corr_df.corr()

print(corr_matrix.mean().mean())

print("\nBTC Correlations:")
print(
    corr_matrix["BTC"]
    .sort_values(ascending=False)
    .head(15)
)

0.49821202556435057

BTC Correlations:
asset
BTC     1.000000
XTZ     0.846731
SUI     0.817376
SOL     0.728694
EOS     0.714804
AVAX    0.693403
LINK    0.689216
UNI     0.688422
DOT     0.683308
BNB     0.645240
SHIB    0.639800
ETH     0.613547
LTC     0.612626
BCH     0.612275
ICP     0.606895
Name: BTC, dtype: float64

## Volume and Liquidity

In [ ]:
vol = data.sel(field="vol")

vol = vol.where(
    data.sel(field="is_liquid") == 1
)

avg_vol = vol.mean("time")

vol_df = avg_vol.to_pandas().sort_values(
    ascending=False
)

print(vol_df.head(15))

print("\nAverage Volume Across Assets:")
print(vol_df.mean())

asset
BTC     2.287320e+10
ETH     1.216280e+10
SOL     3.122761e+09
XRP     2.158502e+09
BCH     1.850242e+09
EOS     1.745038e+09
ZEC     1.616780e+09
LTC     1.481595e+09
LINK    1.466971e+09
BNB     1.457790e+09
SUI     1.293547e+09
DOGE    1.280880e+09
ADA     1.075409e+09
DOT     1.068360e+09
BSV     9.761785e+08
Name: cryptodaily, dtype: float64

Average Volume Across Assets:
1656438292.109125


## Market Regimes

In [ ]:
btc = close.sel(asset="BTC")

btc_sma200 = btc.rolling(
    time=200
).mean()

bull_market = btc > btc_sma200

print(
    "Bull Market Fraction:",
    float(bull_market.mean())
)

Bull Market Fraction: 0.5968398372037348